# Method 3: StyleGAN non-inversion, CoModGAN-style (supervised)
Encoder reads (masked image + mask) and a style code from z; a StyleGAN-style generator outputs the full image. Trained on (masked, original) pairs with adversarial + R1 (+ LPIPS) loss. Fast at test time, stochastic.

**Code lives in** `src/methods/stylegan_noninversion.py`. This notebook only drives it: train / run, evaluate, save results.

Status: method file is a stub; the cells below run once it is implemented.

## 0. Setup

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/EricaCleo/DLpastiA.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm pandas
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(os.getcwd(), '| torch', torch.__version__, '| device:', dev)
if dev == 'cpu': print('NO GPU: on Colab use Runtime > Change runtime type > GPU')

## 1. Data

In [ ]:
# Data (run notebook 00 once first, or this downloads it). Needs env var KAGGLE_API_TOKEN.
from pathlib import Path
if not Path('data/raw').exists():
    !{sys.executable} scripts/download_data.py
from src.data import get_datasets, get_loaders, load_config
cfg = load_config()
train_loader, _ = get_loaders(batch_size=16)
train_ds, test_ds = get_datasets(cfg)
print(len(train_ds), len(test_ds))

## 2. Train (long job: run as a batch job on the university GPU, with checkpoint resume)

In [ ]:
from src.methods.stylegan_noninversion import CoModGANInpainter
gan = CoModGANInpainter(in_channels=4, device=dev)
gan.train(train_loader, epochs=100)

## 3. Evaluate

In [ ]:
from src.eval import evaluate, save_result
for mask in ['square', 'one_sided', 'centre']:
    res = evaluate(gan, mask, n_images=200, device=dev, save_dir=f'outputs/samples/{gan.name}_{mask}')
    save_result(gan.name, mask, res)
    print(mask, res)